# 03 — Cruzamento escola × pixel (estatística zonal ponderada)
O raster é uma grade de 30 m. Em vez de amostrar só o pixel onde cai o ponto, cada pixel entra com peso igual à
**fração da sua área que está dentro do buffer** (`exactextract`). Resultado: médias e extremos que não dependem
de o ponto ter caído na rua ou no telhado vizinho.

Também resume a série temporal (cena a cena) do notebook 02: mín, média, máx, amplitude e desvio entre datas.

In [ ]:
!pip -q install exactextract rasterio geopandas

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import glob, rasterio, numpy as np, pandas as pd, geopandas as gpd
from exactextract import exact_extract

tifs = sorted(glob.glob(f'{RAW}/lst/lst_composto*.tif'))
if len(tifs) > 1:   # export do GEE pode vir em blocos → mosaico virtual
    from osgeo import gdal
    gdal.BuildVRT(f'{INT}/lst_composto.vrt', tifs); TIF = f'{INT}/lst_composto.vrt'
else:
    TIF = tifs[0]
ORDEM_GEE = ['LST_min','LST_mean','LST_median','LST_p90','LST_max','LST_stdDev','ANOM_mean','ANOM_p90','NDVI_median']
with rasterio.open(TIF) as r:
    nomes = list(r.descriptions)
if not all(nomes):   # export sem nomes de banda → ordem do composto do notebook 02
    nomes = ORDEM_GEE
print(nomes)

In [ ]:
OPS = ['mean', 'min', 'max', 'stdev', 'count']   # count = soma das frações de pixel (nº "efetivo" de pixels)
res = []
for b in BUFFERS:
    buf = gpd.read_file(f'{INT}/escolas.gpkg', layer=f'buffer_{b}m')
    z = exact_extract(TIF, buf, OPS, include_cols=['cod_esc'], output='pandas')
    # padroniza nomes: band_1_mean → LST_mean_mean …
    ren = {}
    for c in z.columns:
        for i, n in enumerate(nomes):
            for pre in (f'band_{i+1}_', f'{n}_'):
                if c.startswith(pre):
                    ren[c] = f'{n}__{c[len(pre):]}'
    z = z.rename(columns=ren); z['buffer_m'] = b
    res.append(z)
zonal = pd.concat(res, ignore_index=True)
zonal.to_parquet(f'{INT}/zonal_todos_buffers.parquet')
zonal.head()

In [ ]:
# sensibilidade ao tamanho do buffer: correlação de Spearman do LST médio entre buffers
col = [c for c in zonal.columns if c.startswith('LST_mean__mean')][0]
piv = zonal.pivot(index='cod_esc', columns='buffer_m', values=col)
print(piv.corr(method='spearman').round(3))

In [ ]:
# tabela do buffer de referência (média ponderada de cada banda + variação espacial do LST médio no buffer)
z = zonal[zonal.buffer_m == BUFFER_REF].copy()
base = {c: c.replace('__mean', '') for c in z.columns if c.endswith('__mean')}
temp = z[['cod_esc'] + list(base)].rename(columns=base)
lstm = [n for n in nomes if n.startswith('LST_mean')][0]
temp[f'{lstm}_min_espacial'] = z[f'{lstm}__min']
temp[f'{lstm}_max_espacial'] = z[f'{lstm}__max']
temp['n_pixels_efetivos'] = z[f'{lstm}__count']

In [ ]:
# série temporal → variação entre datas
s = pd.read_csv(glob.glob(f'{RAW}/lst/lst_serie_escolas*.csv')[0], dtype={'cod_esc': str})
s = s.dropna(subset=['LST'])
agg = s.groupby('cod_esc').agg(LST_t_min=('LST', 'min'), LST_t_media=('LST', 'mean'), LST_t_max=('LST', 'max'),
                               LST_t_dp=('LST', 'std'), LST_t_p90=('LST', lambda x: x.quantile(.9)),
                               ANOM_t_media=('ANOM', 'mean'), n_cenas=('LST', 'size'))
agg['LST_t_amplitude'] = agg['LST_t_max'] - agg['LST_t_min']
temp['cod_esc'] = temp['cod_esc'].astype(str)
temp = temp.merge(agg.reset_index(), on='cod_esc', how='left')
temp.to_parquet(f'{PRO}/temperatura_escolas.parquet')
temp.describe().T.round(2)